# VBench custom-input scoring for RQ1 clips

Run this on **Colab, A100 GPU, standard memory**. That runtime is about 40 GB of GPU memory and 84 GB of system RAM.

One raw clip is expanded to about 23 GB of frames in system RAM before VBench resizes it. That picks the machine:

- **A100, standard memory.** Enough RAM for the frames. Leave High-RAM off; that only spends more compute units.
- **L4** has about 53 GB of RAM, which is short of the frame buffer.
- **G4** is an RTX PRO 6000 Blackwell. Same generation as the RTX 5060 this already failed on. VBench's CUDA stack is not built for it.
- **H100** runs, and costs several times the A100 for six short clips.
- **Kaggle T4 x2** does not pool into one 32 GB GPU for these metrics, and a TPU cannot run this PyTorch code.

## Colab, nothing else to install

1. Go to [colab.research.google.com](https://colab.research.google.com) → File → Upload notebook → this file.
2. Runtime → Change runtime type → **A100 GPU**. Leave High-RAM unchecked.
3. Runtime → Run all.
4. When the last cell finishes, it downloads `rq1_vbench_csvs.zip`. Put those CSVs in the repo's `data/` folder. Add `human_ratings.csv` from Google Docs later, then run `rq1-aidrama.ipynb`.

This notebook scores the six VBench 1.0 dimensions that apply to these clips: subject consistency, background consistency, motion smoothness, dynamic degree, aesthetic quality, and imaging quality. It also scores VBench 2.0 `Human_Anatomy` into `vbench2_human_anatomy`. A column that already has a score for every clip is left unchanged. Push `data/metric_scores.csv` before a second Colab run so those columns are not scored again.

`Human_Identity` and `Human_Clothes` are not scored. Both require one person, and these drama clips show several. Diversity and Multi-View Consistency need more than one clip per prompt. Those four are listed in `dimension_applicability.csv` and are absent from `metric_scores.csv`. `Human_Anatomy` looks at each person, so these multi-person clips are in scope. The confirmatory association notebook still uses only the six VBench 1.0 columns.

`Human_Anatomy` keeps the timm Colab already has. Before mmdet is installed or imported, `_pil_interp` is aliased to `str_to_pil_interp` when the installed timm only has the new name. `RQ1_INSTALL_DEPS=0` still installs that alias.

The notebook clones the public clip repo and the pinned VBench checkout, installs the Python packages, and downloads weights. Do not run VBench `setup.py`.

Missing scores stay missing. They are never written as zero. This notebook does not invent human enjoyment ratings.

Outputs:

- `clip_manifest.csv` — `video_id`, `path`, `source_id`, `duration`, `source_id_basis`
- `metric_scores.csv` — one row per clip, the six VBench 1.0 columns plus `vbench2_human_anatomy`
- `dimension_applicability.csv` — what ran, and why a dimension was skipped


In [10]:
from pathlib import Path
import json, math, os, platform, re, shutil, subprocess, sys
import numpy as np
import pandas as pd
from IPython.display import display

ON_COLAB = "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle/working").is_dir()
TPU_RUNTIME = any(os.environ.get(name) for name in ("COLAB_TPU_ADDR", "TPU_NAME", "XRT_TPU_CONFIG"))
WORK_ROOT = Path("/content") if ON_COLAB else Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
CLIP_REPO = "https://github.com/DraPraks/glupov.git"
VBENCH_COMMIT = "fd18b3d055cb0fc6f066ca90fe2c3c8cbb698490"
RUN_SCORING = os.environ.get("RQ1_RUN_SCORING", "1") != "0"
HOSTED = ON_COLAB or ON_KAGGLE
INSTALL_SCORING_DEPS = os.environ.get("RQ1_INSTALL_DEPS", "1" if HOSTED else "0") == "1"
SOURCE_ID = "unlabeled_pilot"
SOURCE_ID_BASIS = "provisional_unlabeled"

A100_SWITCH = (
    "Runtime → Change runtime type → A100 GPU. Leave High-RAM off. "
    "Do not pick L4, G4, H100, T4, or a TPU."
)


def system_ram_gb() -> float:
    try:
        import psutil
        return psutil.virtual_memory().total / (1024 ** 3)
    except Exception:
        pass
    meminfo = Path("/proc/meminfo")
    if meminfo.is_file():
        for line in meminfo.read_text(encoding="utf-8", errors="replace").splitlines():
            if line.startswith("MemTotal:"):
                return int(line.split()[1]) / (1024 ** 2)
    return float("nan")


def require_colab_a100() -> None:
    if not RUN_SCORING:
        print("RQ1_RUN_SCORING=0: GPU check skipped.")
        return
    if TPU_RUNTIME:
        raise RuntimeError(
            "This runtime is a TPU. VBench is PyTorch CUDA and will not run here. " + A100_SWITCH
        )
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("No CUDA GPU is visible. " + A100_SWITCH)
    name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    ram_gb = system_ram_gb()
    print(f"GPU: {name}")
    print(f"Compute capability: {major}.{minor}")
    print(f"GPU memory: {vram_gb:.1f} GB")
    print(f"System RAM: {ram_gb:.1f} GB")
    if major >= 12:
        raise RuntimeError(
            f"{name} is Blackwell (compute capability {major}.{minor}). "
            "VBench's CUDA stack is not built for that generation. " + A100_SWITCH
        )
    if ram_gb < 60 or vram_gb < 35:
        raise RuntimeError(
            f"This runtime has {vram_gb:.0f} GB of GPU memory and {ram_gb:.0f} GB of system RAM. "
            "A full-resolution clip needs about 23 GB of system RAM before VBench resizes it. "
            + A100_SWITCH
        )
    if vram_gb >= 70:
        print("Larger than a standard A100. Scoring will run. Standard A100 spends fewer compute units.")
    else:
        print("A100-class GPU confirmed.")


def directory_has_video(root: Path) -> bool:
    if not root.is_dir():
        return False
    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in {".mp4", ".mov", ".mkv", ".webm", ".avi", ".m4v", ".gif"}:
            if {"vbench_raw", "staged_clips", "rq1_outputs", "rq1_outputs_demo"} & set(path.parts):
                continue
            return True
    return False


def ensure_clip_dir() -> Path:
    override = os.environ.get("RQ1_CLIPS_DIR")
    if override:
        path = Path(override).resolve()
        if not directory_has_video(path):
            raise FileNotFoundError(f"RQ1_CLIPS_DIR has no videos: {path}")
        return path
    candidates = [
        Path.cwd() / "dataset",
        Path.cwd().parent / "dataset",
        WORK_ROOT / "glupov" / "dataset",
        WORK_ROOT / "dataset",
    ]
    if ON_KAGGLE:
        candidates.append(Path("/kaggle/input"))
    for candidate in candidates:
        if directory_has_video(candidate):
            return candidate.resolve()
    dest = WORK_ROOT / "glupov"
    print("Cloning clip repo", CLIP_REPO)
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"
    subprocess.run(
        ["git", "clone", "--depth", "1", CLIP_REPO, str(dest)],
        check=True, env=env,
    )
    dataset = dest / "dataset"
    if not directory_has_video(dataset):
        raise FileNotFoundError(f"Clone succeeded but {dataset} has no videos.")
    return dataset.resolve()


require_colab_a100()
CLIPS_DIR = ensure_clip_dir()
if os.environ.get("RQ1_DATA_DIR"):
    DATA_DIR = Path(os.environ["RQ1_DATA_DIR"]).resolve()
elif HOSTED:
    DATA_DIR = (WORK_ROOT / "rq1_data").resolve()
else:
    repo_guess = CLIPS_DIR.parent if CLIPS_DIR.name == "dataset" else Path.cwd()
    DATA_DIR = (repo_guess / "data").resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
if os.environ.get("RQ1_VBENCH_CACHE"):
    CACHE_DIR = Path(os.environ["RQ1_VBENCH_CACHE"]).resolve()
elif HOSTED:
    CACHE_DIR = (WORK_ROOT / "rq1-vbench").resolve()
else:
    CACHE_DIR = (Path.home() / ".cache" / "rq1-vbench").resolve()
REPO_ROOT = CLIPS_DIR.parent if CLIPS_DIR.name == "dataset" else CLIPS_DIR.parent

VBENCH_DIMENSIONS = [
    "subject_consistency",
    "background_consistency",
    "motion_smoothness",
    "dynamic_degree",
    "aesthetic_quality",
    "imaging_quality",
]
VBENCH2_DIMENSIONS = ["Human_Anatomy"]

def metric_column(benchmark: str, dimension: str) -> str:
    if benchmark == "vbench":
        return "vbench_" + dimension
    return "vbench2_" + dimension.lower()

DIMENSION_SPECS = [
    {"benchmark": "vbench", "dimension": name, "column": metric_column("vbench", name)}
    for name in VBENCH_DIMENSIONS
] + [
    {"benchmark": "vbench2", "dimension": name, "column": metric_column("vbench2", name)}
    for name in VBENCH2_DIMENSIONS
]
DIMENSIONS = [spec["column"] for spec in DIMENSION_SPECS]
EXCLUDED_DIMENSIONS = [
    {
        "benchmark": "vbench2",
        "dimension": "Human_Identity",
        "single_clip": True,
        "status": "not_applicable_multi_person",
        "reason": (
            "VBench-2.0 anchors identity to one face in the opening frame and "
            "drops a clip with fewer than 20 single-face frames. "
            "These drama clips show several people, so the dimension is not scored."
        ),
    },
    {
        "benchmark": "vbench2",
        "dimension": "Human_Clothes",
        "single_clip": True,
        "status": "not_applicable_multi_person",
        "reason": (
            "VBench-2.0 asks whether only one person is present throughout and "
            "does not score clothes otherwise. These drama clips show several people."
        ),
    },
    {
        "benchmark": "vbench2",
        "dimension": "Diversity",
        "single_clip": False,
        "status": "not_applicable_single_clip",
        "reason": (
            "VBench-2.0 custom input needs >=20 videos for one prompt "
            "(prompt-index.mp4, index 0..19). One raw clip has no Diversity score."
        ),
    },
    {
        "benchmark": "vbench2",
        "dimension": "Multi-View_Consistency",
        "single_clip": False,
        "status": "not_applicable_single_clip",
        "reason": (
            "VBench-2.0 custom input expects multiple views in their own folder. "
            "One raw clip is not a multi-view set."
        ),
    },
]

VIDEO_EXTENSIONS = {".mp4", ".mov", ".mkv", ".webm", ".avi", ".m4v", ".gif"}
MANIFEST_CSV = DATA_DIR / "clip_manifest.csv"
METRICS_CSV = DATA_DIR / "metric_scores.csv"
APPLICABILITY_CSV = DATA_DIR / "dimension_applicability.csv"
VBENCH_RAW_DIR = DATA_DIR / "vbench_raw"
STAGED_DIR = DATA_DIR / "staged_clips"

print("REPO_ROOT =", REPO_ROOT)
print("CLIPS_DIR =", CLIPS_DIR)
print("DATA_DIR =", DATA_DIR)
print("Dimensions:", ", ".join(DIMENSIONS))


GPU: NVIDIA A100-SXM4-80GB
Compute capability: 8.0
GPU memory: 79.3 GB
System RAM: 167.1 GB
Larger than a standard A100. Scoring will run. Standard A100 spends fewer compute units.
REPO_ROOT = /content/glupov
CLIPS_DIR = /content/glupov/dataset
DATA_DIR = /content/rq1_data
Dimensions: vbench_subject_consistency, vbench_background_consistency, vbench_motion_smoothness, vbench_dynamic_degree, vbench_aesthetic_quality, vbench_imaging_quality, vbench2_human_anatomy


## Helpers


In [11]:
def coerce_score(raw):
    if raw is None or raw == "":
        return None
    if isinstance(raw, str):
        if raw.strip().lower() in {"nan", "none", "null"}:
            return None
        try:
            raw = float(raw)
        except ValueError:
            return None
    if isinstance(raw, dict):
        for key in ("score", "video_results", "final_score"):
            if key in raw:
                return coerce_score(raw[key])
        return None
    if isinstance(raw, (list, tuple)):
        if len(raw) == 1:
            return coerce_score(raw[0])
        return None
    try:
        value = float(raw)
    except (TypeError, ValueError):
        return None
    if math.isnan(value) or math.isinf(value):
        return None
    return value


def parse_dimension_payload(payload):
    records = []
    if isinstance(payload, dict):
        records = [payload]
    elif isinstance(payload, list):
        for item in payload:
            if isinstance(item, dict) and ("video_path" in item or "video_results" in item):
                records.append(item)
            elif isinstance(item, list):
                records.extend(sub for sub in item if isinstance(sub, dict))
    found = {}
    for record in records:
        path = record.get("video_path") or record.get("video_name") or ""
        found[str(path)] = coerce_score(record.get("video_results", None))
    return found


def file_name_of(path_str: str) -> str:
    return str(path_str).replace("\\", "/").rstrip("/").split("/")[-1]


def lookup_score(parsed: dict, staged_name: str):
    staged_stem = Path(staged_name).stem
    for key, value in parsed.items():
        name = file_name_of(key)
        if name == staged_name or Path(name).stem == staged_stem:
            return value
    return None


def list_videos(root: Path):
    root = Path(root)
    if root.is_file() and root.suffix.lower() in VIDEO_EXTENSIONS:
        return [root]
    if not root.is_dir():
        return []
    found = []
    for path in sorted(root.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in VIDEO_EXTENSIONS:
            continue
        if {"data", "vbench_raw", "staged_clips", "rq1_outputs"} & set(path.parts):
            continue
        found.append(path)
    return found


def probe_duration(path: Path):
    ffprobe = shutil.which("ffprobe")
    if ffprobe:
        proc = subprocess.run(
            [ffprobe, "-v", "error", "-show_entries", "format=duration",
             "-of", "default=noprint_wrappers=1:nokey=1", str(path)],
            capture_output=True, text=True,
        )
        if proc.returncode == 0 and proc.stdout.strip():
            try:
                return float(proc.stdout.strip()), ""
            except ValueError:
                pass
    try:
        import cv2
        capture = cv2.VideoCapture(str(path))
        fps = capture.get(cv2.CAP_PROP_FPS) or 0
        frames = capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0
        capture.release()
        if fps > 0 and frames > 0:
            return float(frames) / float(fps), ""
    except Exception:
        pass
    return float("nan"), "Duration unread (need ffprobe or cv2)."


def make_video_id(stem: str, used: set[str]) -> str:
    base = re.sub(r"[^A-Za-z0-9._-]+", "_", stem).strip("._") or "clip"
    video_id = base
    n = 2
    while video_id in used:
        video_id = f"{base}_{n}"
        n += 1
    used.add(video_id)
    return video_id


def applicability_table(per_dimension: dict | None = None) -> pd.DataFrame:
    per_dimension = per_dimension or {}
    rows = []
    for spec in DIMENSION_SPECS:
        extra = per_dimension.get(spec["dimension"], {})
        rows.append({
            "dimension": spec["dimension"],
            "column_name": spec["column"],
            "benchmark": spec["benchmark"],
            "custom_input_single_clip": True,
            "status": extra.get("status", "not_run"),
            "reason": extra.get(
                "reason",
                "Official custom_input scores one raw video on this dimension.",
            ),
        })
    for spec in EXCLUDED_DIMENSIONS:
        rows.append({
            "dimension": spec["dimension"],
            "column_name": "",
            "benchmark": spec["benchmark"],
            "custom_input_single_clip": spec["single_clip"],
            "status": spec["status"],
            "reason": spec["reason"],
        })
    return pd.DataFrame(rows)


def pip_would_change_torch(packages):
    proc = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--dry-run", *packages],
        capture_output=True, text=True,
    )
    text = ((proc.stdout or "") + "\n" + (proc.stderr or "")).lower()
    return "uninstalling torch" in text or "would install torch" in text or "downloading torch" in text


def install_packages(packages):
    if pip_would_change_torch(packages):
        raise RuntimeError(
            "Refusing to install " + ", ".join(packages)
            + " because pip would change the current torch build."
        )
    subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])


def ensure_cuda_torch():
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("PyTorch CUDA is not available. " + A100_SWITCH)
    return f"CUDA torch {torch.__version__} on {torch.cuda.get_device_name(0)}"


def ensure_decord():
    try:
        import decord  # noqa: F401
        return "decord import ok"
    except Exception as first_exc:
        if not INSTALL_SCORING_DEPS:
            raise RuntimeError(
                "decord is required for VBench video loading "
                f"({first_exc}). Set RQ1_INSTALL_DEPS=1 and rerun."
            ) from first_exc
        last_exc = first_exc
        for package in ("decord", "eva-decord"):
            try:
                install_packages([package])
                import decord  # noqa: F401
                return package + " installed"
            except Exception as exc:
                last_exc = exc
        raise RuntimeError(f"Could not install decord or eva-decord ({last_exc})") from last_exc


def shim_torchvision_write_video():
    import torchvision.io as video_io
    if not hasattr(video_io, "write_video"):
        def write_video(*args, **kwargs):
            raise RuntimeError(
                "torchvision.io.write_video is not available in this torchvision build"
            )
        video_io.write_video = write_video
        note = (
            "Added a torchvision.io.write_video shim. "
            "VBench-2.0 imports that name and does not call it."
        )
    else:
        note = "torchvision.io.write_video present"
    for name in list(sys.modules):
        if name == "vbench2" or name.startswith("vbench2."):
            del sys.modules[name]
    return note


def load_saved_metrics():
    candidates = [METRICS_CSV, REPO_ROOT / "data" / "metric_scores.csv", WORK_ROOT / "metric_scores.csv", Path.cwd() / "metric_scores.csv"]
    found = []
    seen = set()
    for path in candidates:
        try:
            resolved = path.resolve()
        except OSError:
            continue
        if resolved in seen or not path.is_file():
            continue
        seen.add(resolved)
        frame = pd.read_csv(path, dtype={"video_id": str, "source_id": str})
        frame["video_id"] = frame["video_id"].astype(str)
        filled = 0
        for column in DIMENSIONS:
            if column in frame.columns:
                filled += int(pd.to_numeric(frame[column], errors="coerce").notna().sum())
        found.append((filled, path, frame))
        print("Found saved scores", path, "filled cells", filled)
    if not found:
        return None
    found.sort(key=lambda item: item[0], reverse=True)
    return found[0][2]


def install_core_packages():
    if not INSTALL_SCORING_DEPS:
        print("RQ1_INSTALL_DEPS=0: using packages already installed.")
        return
    required = [
        "opencv-python", "pyyaml", "easydict", "omegaconf", "tqdm",
        "einops", "timm", "scikit-image", "Pillow", "fvcore", "iopath",
        "ftfy", "regex", "openai-clip", "pyiqa", "gdown",
        "imageio", "imageio-ffmpeg", "huggingface_hub", "transformers==4.51.0",
        "tokenizers>=0.21,<0.22",
        "accelerate", "sentencepiece", "safetensors", "fairscale", "scenedetect",
    ]
    skipped = []
    for package in required:
        if pip_would_change_torch([package]):
            skipped.append(package)
            print("Skipping", package, "because pip would replace the CUDA torch build.")
            continue
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
    if skipped:
        print("Packages left unchanged:", ", ".join(skipped))


ANATOMY_YOLO_COMMIT = "4826695a84ce370dad7fe4cb555a6c7162f3751d"
ANATOMY_YOLO_URL = "git+https://github.com/AILab-CVC/YOLO-World.git@" + ANATOMY_YOLO_COMMIT
ANATOMY_PACKAGES = [
    "setuptools>=77",
    "mmengine==0.10.5",
    "mmdet==3.0.0",
    "mmyolo==0.6.0",
    "supervision==0.19.0",
]
def anatomy_mmcv_requirement():
    import torch
    cuda = torch.version.cuda
    if not cuda or not torch.cuda.is_available():
        raise RuntimeError("Human Anatomy needs the CUDA torch that is already installed. " + A100_SWITCH)
    major_minor = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    version = f"2.2.0+cu.{cuda}.torch.{major_minor}"
    index = "https://wheels.astral.sh/simple/cu" + cuda.replace(".", "") + "/"
    return version, index


def install_pinned(packages):
    pending = []
    for package in packages:
        if pip_would_change_torch([package]):
            print("Skipping", package, "because pip would replace the CUDA torch build.")
            continue
        pending.append(package)
    if pending:
        subprocess.check_call([sys.executable, "-m", "pip", "install", *pending])


def install_anatomy_mmcv():
    import importlib.metadata
    version, index = anatomy_mmcv_requirement()
    try:
        installed = importlib.metadata.version("mmcv")
    except importlib.metadata.PackageNotFoundError:
        installed = ""
    if installed == version:
        print("mmcv", version, "already installed")
        return
    spec = "mmcv==" + version
    command = [sys.executable, "-m", "pip", "install", spec, "--extra-index-url", index]
    probe = subprocess.run(command + ["--dry-run"], capture_output=True, text=True)
    probe_text = ((probe.stdout or "") + "\n" + (probe.stderr or "")).lower()
    if "uninstalling torch" in probe_text or "would install torch" in probe_text:
        command.append("--no-deps")
        print("Installing", spec, "without dependencies so the CUDA torch build stays.")
    else:
        print("Installing", spec)
    subprocess.check_call(command)


def installed_yolo_commit():
    import importlib.metadata
    try:
        dist = importlib.metadata.distribution("yolo-world")
    except importlib.metadata.PackageNotFoundError:
        return ""
    raw = dist.read_text("direct_url.json") or ""
    if not raw:
        return ""
    try:
        payload = json.loads(raw)
    except json.JSONDecodeError:
        return ""
    vcs = payload.get("vcs_info") or {}
    return str(vcs.get("commit_id") or "")


def install_anatomy_yolo():
    found = installed_yolo_commit()
    if found == ANATOMY_YOLO_COMMIT:
        print("yolo-world", ANATOMY_YOLO_COMMIT, "already installed")
        return
    print("Installing YOLO-World", ANATOMY_YOLO_COMMIT, "without replacing mmyolo.")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-deps",
        "--no-build-isolation", ANATOMY_YOLO_URL,
    ])
    found = installed_yolo_commit()
    if found != ANATOMY_YOLO_COMMIT:
        raise RuntimeError(
            "YOLO-World install did not record commit " + ANATOMY_YOLO_COMMIT
            + ". Found " + (found or "no direct URL") + "."
        )
    if "yolo_world" in sys.modules:
        raise RuntimeError(
            "YOLO-World was reinstalled after it had been imported. "
            "Runtime → Restart runtime, then Run all."
        )


def patch_anatomy_packages():
    import site
    cap_old = "mmcv_maximum_version = '2.1.0'"
    cap_new = "mmcv_maximum_version = '2.3.0'"
    adafactor_old = "OPTIMIZERS.register_module(name='Adafactor', module=Adafactor)"
    adafactor_new = "OPTIMIZERS.register_module(name='Adafactor', module=Adafactor, force=True)"
    notes = []
    loaded = [
        name for name in ("mmdet", "mmyolo", "mmengine.optim.optimizer.builder", "yolo_world")
        if name in sys.modules
    ]
    for root in site.getsitepackages():
        for rel in ("mmdet/__init__.py", "mmyolo/__init__.py"):
            path = Path(root, rel)
            if path.is_file() and cap_old in path.read_text(encoding="utf-8"):
                path.write_text(path.read_text(encoding="utf-8").replace(cap_old, cap_new), encoding="utf-8")
                notes.append(str(path))
        builder = Path(root, "mmengine/optim/optimizer/builder.py")
        if builder.is_file() and adafactor_old in builder.read_text(encoding="utf-8"):
            text = builder.read_text(encoding="utf-8")
            builder.write_text(text.replace(adafactor_old, adafactor_new, 1), encoding="utf-8")
            notes.append(str(builder))
    if notes and loaded:
        raise RuntimeError(
            "Anatomy patches were written after "
            + ", ".join(loaded)
            + " had already been imported. Runtime → Restart runtime, then Run all."
        )
    if notes:
        print("Patched anatomy packages:")
        print("\n".join(notes))
    else:
        print("Anatomy package patches already present.")


def patch_anatomy_clip_config(checkout: Path):
    root = checkout / "VBench-2.0" / "vbench2" / "third_party" / "YOLO-World"
    private_clip = "/mnt/petrelfs/zhengdian/code/ckpt/clip-vit-base-patch32"
    public_clip = "openai/clip-vit-base-patch32"
    if not root.is_dir():
        raise RuntimeError(f"Missing YOLO-World configs under {root}")
    changed = []
    for path in sorted(root.rglob("*.py")):
        text = path.read_text(encoding="utf-8")
        if private_clip in text:
            path.write_text(text.replace(private_clip, public_clip), encoding="utf-8")
            changed.append(str(path.relative_to(root)))
    remaining = []
    private_bytes = private_clip.encode("utf-8")
    for path in sorted(root.rglob("*")):
        if path.is_file() and private_bytes in path.read_bytes():
            remaining.append(str(path.relative_to(root)))
    if remaining:
        raise RuntimeError("Private CLIP path remains in " + ", ".join(remaining))
    if changed:
        print("Pointed CLIP at openai/clip-vit-base-patch32 in", ", ".join(changed))
    else:
        print("YOLO-World CLIP path already points at a public model.")


def shim_timm_pil_interp():
    # _pil_interp was renamed to str_to_pil_interp. Alias it before mmdet imports the old name.
    import timm.data.transforms as t
    if not hasattr(t, "_pil_interp"):
        t._pil_interp = t.str_to_pil_interp
    if not hasattr(t, "_pil_interp"):
        raise RuntimeError(
            "timm.data.transforms._pil_interp is missing after aliasing str_to_pil_interp."
        )


def install_anatomy_stack(checkout: Path):
    shim_timm_pil_interp()
    if not INSTALL_SCORING_DEPS:
        print("RQ1_INSTALL_DEPS=0: expecting the Human Anatomy packages to be installed.")
        patch_anatomy_packages()
        patch_anatomy_clip_config(checkout)
        return
    install_pinned(ANATOMY_PACKAGES)
    install_anatomy_mmcv()
    install_anatomy_yolo()
    patch_anatomy_packages()
    patch_anatomy_clip_config(checkout)


def ensure_vbench_checkout():
    dest = CACHE_DIR / "VBench"
    dest.parent.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"
    if not (dest / ".git").is_dir():
        if dest.exists():
            raise RuntimeError(f"{dest} exists but is not a git checkout.")
        subprocess.run(
            ["git", "clone", "https://github.com/Vchitect/VBench.git", str(dest)],
            check=True, env=env,
        )
    head = subprocess.run(
        ["git", "-C", str(dest), "rev-parse", "HEAD"],
        capture_output=True, text=True, env=env,
    )
    current = (head.stdout or "").strip()
    if current != VBENCH_COMMIT:
        subprocess.run(
            ["git", "-C", str(dest), "fetch", "--depth", "1", "origin", VBENCH_COMMIT],
            check=True, env=env,
        )
        subprocess.run(
            ["git", "-C", str(dest), "checkout", VBENCH_COMMIT],
            check=True, env=env,
        )
        current = VBENCH_COMMIT
    return dest, current


def stage_for_vbench(src: Path, dest_mp4: Path) -> Path:
    dest_mp4.parent.mkdir(parents=True, exist_ok=True)
    suffix = src.suffix.lower()
    if suffix == ".gif":
        gif_dest = dest_mp4.with_suffix(".gif")
        if src.resolve() != gif_dest.resolve():
            shutil.copy2(src, gif_dest)
        return gif_dest
    if suffix == ".mp4":
        if src.resolve() != dest_mp4.resolve():
            shutil.copy2(src, dest_mp4)
        return dest_mp4
    ffmpeg = shutil.which("ffmpeg")
    if not ffmpeg:
        raise RuntimeError(
            f"{src.name} is {suffix}. custom_input reads .mp4/.gif only, "
            "and ffmpeg is not on PATH."
        )
    copy = subprocess.run(
        [ffmpeg, "-y", "-i", str(src), "-c", "copy", str(dest_mp4)],
        capture_output=True, text=True,
    )
    if copy.returncode != 0:
        encode = subprocess.run(
            [ffmpeg, "-y", "-i", str(src), "-c:v", "libx264", "-pix_fmt", "yuv420p",
             "-an", str(dest_mp4)],
            capture_output=True, text=True,
        )
        if encode.returncode != 0:
            raise RuntimeError((encode.stderr or copy.stderr or "ffmpeg failed")[-800:])
    return dest_mp4


# Parser self-check: a returned 0 stays 0; missing stays missing.
_sample = [
    {"video_path": "a.mp4", "video_results": 0.0},
    {"video_path": "b.mp4", "video_results": None},
]
_parsed = parse_dimension_payload(_sample)
assert _parsed["a.mp4"] == 0.0
assert _parsed["b.mp4"] is None
print("Helpers ready.")


Helpers ready.


## 1. Build clip manifest

Pilot `source_id` is the shared provisional value `unlabeled_pilot`. Replace with real drama IDs before confirmatory independent-clip inference.


In [12]:
videos = list_videos(CLIPS_DIR)
if not videos:
    raise FileNotFoundError(
        f"No videos under {CLIPS_DIR}. Set RQ1_CLIPS_DIR or put mp4 files in dataset/."
    )

previous = {}
if MANIFEST_CSV.exists():
    old = pd.read_csv(MANIFEST_CSV, dtype=str)
    for record in old.to_dict("records"):
        if record.get("path"):
            previous[str(Path(record["path"]).resolve())] = record

used_ids: set[str] = set()
rows = []
duration_notes = []
for path in videos:
    path_str = str(path.resolve())
    prior = previous.get(path_str, {})
    if prior.get("video_id"):
        video_id = str(prior["video_id"])
        used_ids.add(video_id)
    else:
        video_id = make_video_id(path.stem, used_ids)
    # Prefer an already-edited source_id; otherwise keep the provisional shared id.
    if prior.get("source_id") and prior.get("source_id_basis") != SOURCE_ID_BASIS:
        source_id = str(prior["source_id"])
        basis = str(prior.get("source_id_basis") or "edited_manifest")
    elif prior.get("source_id") and str(prior.get("source_id")) != SOURCE_ID:
        source_id = str(prior["source_id"])
        basis = "edited_manifest"
    else:
        source_id = SOURCE_ID
        basis = SOURCE_ID_BASIS
    duration, note = probe_duration(path)
    if note:
        duration_notes.append(f"{video_id}: {note}")
    try:
        rel_path = str(path.resolve().relative_to(REPO_ROOT))
    except ValueError:
        rel_path = path_str
    rows.append({
        "video_id": video_id,
        "path": rel_path.replace("\\", "/"),
        "source_id": source_id,
        "duration": duration,
        "source_id_basis": basis,
    })

manifest = pd.DataFrame(rows)
manifest.to_csv(MANIFEST_CSV, index=False)
print(f"Wrote {MANIFEST_CSV} ({len(manifest)} clips)")
if duration_notes:
    print("Duration notes:")
    print("\n".join(duration_notes))
display(manifest)


Wrote /content/rq1_data/clip_manifest.csv (11 clips)


,video_id,path,source_id,duration,source_id_basis
0,drama_clip_01,dataset/drama_clip_01.mp4,unlabeled_pilot,32.019308,provisional_unlabeled
1,drama_clip_02,dataset/drama_clip_02.mp4,unlabeled_pilot,15.985442,provisional_unlabeled
2,drama_clip_03,dataset/drama_clip_03.mp4,unlabeled_pilot,29.034667,provisional_unlabeled
3,drama_clip_04,dataset/drama_clip_04.mp4,unlabeled_pilot,23.680000,provisional_unlabeled
4,drama_clip_05,dataset/drama_clip_05.mp4,unlabeled_pilot,33.000000,provisional_unlabeled
5,drama_clip_06,dataset/drama_clip_06.mp4,unlabeled_pilot,31.000000,provisional_unlabeled
6,drama_clip_07,dataset/drama_clip_07.mp4,unlabeled_pilot,21.525333,provisional_unlabeled
7,drama_clip_08,dataset/drama_clip_08.mp4,unlabeled_pilot,38.912000,provisional_unlabeled
8,drama_clip_09,dataset/drama_clip_09.mp4,unlabeled_pilot,21.546667,provisional_unlabeled
9,drama_clip_10,dataset/drama_clip_10.mp4,unlabeled_pilot,29.717333,provisional_unlabeled


## 2. Official custom-input scoring

Each VBench 1.0 dimension, and `Human_Anatomy`, is submitted alone with `mode='custom_input'`. Identity, clothes, Diversity, and Multi-View Consistency are not submitted. Set `RQ1_RUN_SCORING=0` to rebuild the metric table skeleton / reuse an existing file without calling VBench again.


In [13]:
!pip install -q numpy cython
!pip install -q --no-build-isolation "git+https://github.com/lvis-dataset/lvis-api.git"

import timm.data.transforms as t
if not hasattr(t, "_pil_interp"):
    t._pil_interp = t.str_to_pil_interp

import lvis
print("lvis", lvis.__file__)

  Preparing metadata (pyproject.toml) ... done
lvis /usr/local/lib/python3.13/dist-packages/lvis/__init__.py


In [14]:
def blank_metric_frame(manifest_df: pd.DataFrame) -> pd.DataFrame:
    frame = manifest_df[["video_id", "source_id"]].copy()
    frame["video_id"] = frame["video_id"].astype(str)
    frame["source_id"] = frame["source_id"].astype(str)
    for column in DIMENSIONS:
        frame[column] = np.nan
    return frame


def write_metric_frame(frame: pd.DataFrame) -> None:
    ordered = ["video_id", "source_id", *DIMENSIONS]
    out = frame[ordered].copy()
    for column in DIMENSIONS:
        out[column] = pd.to_numeric(out[column], errors="coerce")
    out.to_csv(METRICS_CSV, index=False)


def score_one(benchmark, dimension, column, staged_by_id, full_json):
    import torch
    run_name = column
    VBENCH_RAW_DIR.mkdir(parents=True, exist_ok=True)
    previous_cwd = Path.cwd()
    try:
        if benchmark == "vbench2":
            os.chdir(VBENCH2_HOME)
        if benchmark == "vbench":
            from vbench import VBench
            bench = VBench("cuda", str(full_json), str(VBENCH_RAW_DIR))
        else:
            from vbench2 import VBench2
            bench = VBench2("cuda", str(full_json), str(VBENCH_RAW_DIR))
        kwargs = dict(
            videos_path=str(STAGED_DIR),
            name=run_name,
            dimension_list=[dimension],
            mode="custom_input",
            local=False,
            read_frame=False,
            prompt_list=[],
        )
        if dimension == "imaging_quality":
            kwargs["imaging_quality_preprocessing_mode"] = "longer"
        bench.evaluate(**kwargs)
    finally:
        os.chdir(previous_cwd)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    result_path = VBENCH_RAW_DIR / f"{run_name}_eval_results.json"
    if not result_path.is_file():
        raise RuntimeError(f"evaluate() did not write {result_path}")
    payload = json.loads(result_path.read_text(encoding="utf-8"))
    if dimension not in payload:
        raise RuntimeError(f"{result_path} has no '{dimension}' entry. Keys: {list(payload)}")
    parsed = parse_dimension_payload(payload[dimension])
    per_video = {}
    missing = []
    for video_id, staged in staged_by_id.items():
        value = lookup_score(parsed, staged.name) if parsed else None
        per_video[video_id] = value
        if value is None:
            missing.append(video_id)
    reason = f"Read per-video custom_input scores from {result_path.name}."
    if missing:
        reason += " No score returned for: " + ", ".join(missing) + "."
    if not parsed:
        reason = f"{result_path.name} had no per-video score records."
    return per_video, reason


metrics = blank_metric_frame(manifest)
saved_metrics = load_saved_metrics()
if saved_metrics is not None:
    saved_indexed = saved_metrics.drop_duplicates("video_id").set_index("video_id")
    for column in DIMENSIONS:
        if column in saved_indexed.columns:
            metrics[column] = metrics["video_id"].map(
                pd.to_numeric(saved_indexed[column], errors="coerce")
            )
per_dimension: dict = {}
blocker = ""

if not RUN_SCORING:
    if METRICS_CSV.exists():
        old = pd.read_csv(METRICS_CSV, dtype={"video_id": str, "source_id": str})
        for column in DIMENSIONS:
            if column in old.columns:
                metrics[column] = metrics["video_id"].map(old.set_index("video_id")[column])
        print("RQ1_RUN_SCORING=0: reused", METRICS_CSV)
    else:
        write_metric_frame(metrics)
        print("RQ1_RUN_SCORING=0: wrote empty metric skeleton", METRICS_CSV)
else:
    try:
        if not shutil.which("wget") or not shutil.which("unzip"):
            raise RuntimeError(
                "Official VBench weight download calls wget and unzip. "
                "Both exist on Colab. " + A100_SWITCH
            )
        torch_note = ensure_cuda_torch()
        checkout, commit = ensure_vbench_checkout()
        print(torch_note)
        print("VBench checkout", commit)
        entry = str(checkout)
        if entry not in sys.path:
            sys.path.insert(0, entry)
        os.environ["VBENCH_CACHE_DIR"] = str(CACHE_DIR / "weights")
        os.environ["VBENCH2_CACHE_DIR"] = str(CACHE_DIR / "weights")
        os.environ["TOKENIZERS_PARALLELISM"] = "false"
        VBENCH2_HOME = checkout / "VBench-2.0"
        vbench2_entry = str(VBENCH2_HOME)
        if vbench2_entry not in sys.path:
            sys.path.insert(0, vbench2_entry)
        print(shim_torchvision_write_video())
        install_core_packages()
        print(ensure_decord())
        full_vbench = checkout / "vbench" / "VBench_full_info.json"
        full_vbench2 = VBENCH2_HOME / "vbench2" / "VBench2_full_info.json"
        STAGED_DIR.mkdir(parents=True, exist_ok=True)
        staged_by_id = {}
        stage_errors = []
        for record in manifest.itertuples(index=False):
            src = (REPO_ROOT / record.path).resolve() if not Path(record.path).is_absolute() else Path(record.path)
            try:
                staged_by_id[str(record.video_id)] = stage_for_vbench(
                    src, STAGED_DIR / f"{record.video_id}.mp4"
                )
            except Exception as exc:
                stage_errors.append(f"{record.video_id}: {exc}")
        if stage_errors:
            print("Staging issues:\n" + "\n".join(stage_errors))
        if not staged_by_id:
            raise RuntimeError("No clips could be staged for VBench.")

        for spec in DIMENSION_SPECS:
            full_json = full_vbench if spec["benchmark"] == "vbench" else full_vbench2
            if not full_json.is_file():
                per_dimension[spec["dimension"]] = {
                    "status": "failed",
                    "reason": f"Missing full-info JSON: {full_json}",
                }
                continue
            already = pd.to_numeric(metrics[spec["column"]], errors="coerce")
            if already.notna().all():
                per_dimension[spec["dimension"]] = {
                    "status": "scored",
                    "reason": "Reused complete scores already stored in metric_scores.csv.",
                }
                print(" ", spec["dimension"], "already scored, left unchanged")
                continue
            try:
                if spec["dimension"] == "Human_Anatomy":
                    install_anatomy_stack(checkout)
                print("Scoring", spec["dimension"], "...")
                per_video, reason = score_one(
                    spec["benchmark"], spec["dimension"], spec["column"],
                    staged_by_id, full_json,
                )
                for video_id, value in per_video.items():
                    metrics.loc[metrics["video_id"] == video_id, spec["column"]] = (
                        np.nan if value is None else value
                    )
                per_dimension[spec["dimension"]] = {"status": "scored", "reason": reason}
                print(" ", spec["dimension"], "ok")
            except Exception as exc:
                msg = (type(exc).__name__ + ": " + str(exc))[-1000:]
                per_dimension[spec["dimension"]] = {"status": "failed", "reason": msg}
                print(" ", spec["dimension"], "FAILED:", msg[:300])
            write_metric_frame(metrics)
    except Exception as exc:
        blocker = str(exc)
        print("Scoring blocked:", blocker)
        write_metric_frame(metrics)

applicability_table(per_dimension).to_csv(APPLICABILITY_CSV, index=False)
write_metric_frame(metrics)
print("Wrote", METRICS_CSV)
print("Wrote", APPLICABILITY_CSV)
display(metrics)
display(pd.read_csv(APPLICABILITY_CSV))
if blocker:
    print("BLOCKER:", blocker)


Found saved scores /content/rq1_data/metric_scores.csv filled cells 66
Found saved scores /content/glupov/data/metric_scores.csv filled cells 36
CUDA torch 2.14.1+cu130 on NVIDIA A100-SXM4-80GB
VBench checkout fd18b3d055cb0fc6f066ca90fe2c3c8cbb698490
torchvision.io.write_video present
decord import ok
  subject_consistency already scored, left unchanged
  background_consistency already scored, left unchanged
  motion_smoothness already scored, left unchanged
  dynamic_degree already scored, left unchanged
  aesthetic_quality already scored, left unchanged
  imaging_quality already scored, left unchanged
mmcv 2.2.0+cu.13.0.torch.2.14 already installed
yolo-world 4826695a84ce370dad7fe4cb555a6c7162f3751d already installed
Anatomy package patches already present.
YOLO-World CLIP path already points at a public model.
Scoring Human_Anatomy ...
Evaluation meta data saved to /content/rq1_data/vbench_raw/vbench2_human_anatomy_full_info.json
cur_full_info_path: /content/rq1_data/vbench_raw/vben

100%|██████████| 11/11 [15:38<00:00, 85.34s/it]

Evaluation results saved to /content/rq1_data/vbench_raw/vbench2_human_anatomy_eval_results.json
  Human_Anatomy ok
Wrote /content/rq1_data/metric_scores.csv
Wrote /content/rq1_data/dimension_applicability.csv


,video_id,source_id,vbench_subject_consistency,vbench_background_consistency,vbench_motion_smoothness,vbench_dynamic_degree,vbench_aesthetic_quality,vbench_imaging_quality,vbench2_human_anatomy
0,drama_clip_01,unlabeled_pilot,0.773768,0.888017,0.980377,1.0,0.595418,74.399546,0.945489
1,drama_clip_02,unlabeled_pilot,0.703869,0.801868,0.975042,1.0,0.465960,50.013496,0.964770
2,drama_clip_03,unlabeled_pilot,0.837521,0.887774,0.991490,1.0,0.623501,65.888132,0.998626
3,drama_clip_04,unlabeled_pilot,0.716336,0.865210,0.989857,1.0,0.499387,70.012719,0.998758
4,drama_clip_05,unlabeled_pilot,0.617022,0.777246,0.989038,1.0,0.510078,70.247849,0.929163
5,drama_clip_06,unlabeled_pilot,0.795346,0.846003,0.983371,1.0,0.442541,64.661273,0.617284
6,drama_clip_07,unlabeled_pilot,0.818162,0.838304,0.989680,1.0,0.553206,63.461760,1.000000
7,drama_clip_08,unlabeled_pilot,0.728325,0.847209,0.980504,1.0,0.506821,53.623558,0.984601
8,drama_clip_09,unlabeled_pilot,0.755278,0.862818,0.978702,1.0,0.556560,66.978114,0.998322
9,drama_clip_10,unlabeled_pilot,0.747616,0.827560,0.984354,1.0,0.498573,65.404758,0.923656


,dimension,column_name,benchmark,custom_input_single_clip,status,reason
0,subject_consistency,vbench_subject_consistency,vbench,True,scored,Reused complete scores already stored in metri...
1,background_consistency,vbench_background_consistency,vbench,True,scored,Reused complete scores already stored in metri...
2,motion_smoothness,vbench_motion_smoothness,vbench,True,scored,Reused complete scores already stored in metri...
3,dynamic_degree,vbench_dynamic_degree,vbench,True,scored,Reused complete scores already stored in metri...
4,aesthetic_quality,vbench_aesthetic_quality,vbench,True,scored,Reused complete scores already stored in metri...
5,imaging_quality,vbench_imaging_quality,vbench,True,scored,Reused complete scores already stored in metri...
6,Human_Anatomy,vbench2_human_anatomy,vbench2,True,scored,Read per-video custom_input scores from vbench...
7,Human_Identity,NaN,vbench2,True,not_applicable_multi_person,VBench-2.0 anchors identity to one face in the...
8,Human_Clothes,NaN,vbench2,True,not_applicable_multi_person,VBench-2.0 asks whether only one person is pre...
9,Diversity,NaN,vbench2,False,not_applicable_single_clip,VBench-2.0 custom input needs >=20 videos for ...


## 3. Download the CSVs

The zip contains `clip_manifest.csv`, `metric_scores.csv`, and `dimension_applicability.csv`. On Colab the last lines start a browser download. Unzip into the repo's `data/` folder.

Human enjoyment is not in this zip. When the Google Docs ratings exist, add `data/human_ratings.csv` with columns `video_id`, `rater_id`, `enjoyment` (integers 1–5), replace `unlabeled_pilot` with real drama IDs, then run `notebooks/rq1-aidrama.ipynb`.


In [15]:
import zipfile
zip_path = DATA_DIR.parent / "rq1_vbench_csvs.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in ("clip_manifest.csv", "metric_scores.csv", "dimension_applicability.csv"):
        path = DATA_DIR / name
        if path.is_file():
            archive.write(path, arcname=name)
            print("packed", name)
print("Wrote", zip_path)
if ON_COLAB:
    from google.colab import files
    files.download(str(zip_path))
else:
    print("CSVs are in", DATA_DIR)


packed clip_manifest.csv
packed metric_scores.csv
packed dimension_applicability.csv
Wrote /content/rq1_vbench_csvs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>